# Histograms with atomics and a block scan

This notebook goes with chapter 5.5 (scan and histogram) and chapter 7.2 (privatisation).
You will build a histogram of byte values two ways, with global atomics and with a private copy per block in shared
memory, and then write the work efficient Blelloch scan for one block.

Run it in Google Colab with a T4 GPU, or with NUMBA_ENABLE_CUDASIM=1 for small sizes on the CPU.

In [ ]:
import os
import numpy as np
from numba import cuda, int32

SIM = os.environ.get("NUMBA_ENABLE_CUDASIM") == "1"
N = 3_000 if SIM else 50_000_000
BINS = 256
TPB = 128 if SIM else 256
data = np.random.default_rng(3).integers(0, BINS, N).astype(np.uint8)
d_data = cuda.to_device(data)
ref = np.bincount(data, minlength=BINS)
print("N =", N, " most common value:", ref.argmax(), "seen", ref.max(), "times")

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>import os</code> | The simulator switch. |
| 2 | <code>import numpy as np</code> | NumPy for the data and the check. |
| 3 | <code>from numba import cuda, int32</code> | int32 is the type of the shared memory histogram. |
| 5 | <code>SIM = os.environ.get(&quot;NUMBA_ENABLE_CUDASIM&quot;) == &quot;1&quot;</code> | True in the simulator. |
| 6 | <code>N = 3_000 if SIM else 50_000_000</code> | Fifty million bytes on a GPU, a few thousand in the simulator. |
| 7 | <code>BINS = 256</code> | One bin for each byte value 0 to 255. |
| 8 | <code>TPB = 128 if SIM else 256</code> | Threads per block. |
| 9 | <code>data = np.random.default_rng(3).integers(0, BINS, N).astype(np.uint8)</code> | Random bytes with a fixed seed. In a real program this could be the pixels of an image. |
| 10 | <code>d_data = cuda.to_device(data)</code> | Copy the data to the GPU. |
| 11 | <code>ref = np.bincount(data, minlength=BINS)</code> | NumPy counts how often each value appears. This is our reference. |
| 12 | <code>print(&quot;N =&quot;, N, &quot; most common value:&quot;, ref.argmax(), &quot;seen&quot;, ref.max(), &quot;times&quot;)</code> | Prints one fact about the data as a sanity check. |

## 1. Global atomics

Every thread adds 1 to the bin of its byte. Two threads with the same byte would race on a plain += , so we use an
atomic add. With only 256 bins and millions of threads, many atomics hit the same addresses and wait for each other.

In [ ]:
@cuda.jit
def hist_global(data, hist):
    i = cuda.grid(1)
    stride = cuda.gridsize(1)
    while i < data.size:
        cuda.atomic.add(hist, data[i], 1)
        i += stride

hist = cuda.to_device(np.zeros(BINS, dtype=np.int32))
blocks = 30 if SIM else 1024
hist_global[blocks, TPB](d_data, hist)
print("global atomics correct:", np.array_equal(hist.copy_to_host(), ref))

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>@cuda.jit</code> | Compile as a kernel. |
| 2 | <code>def hist_global(data, hist):</code> | data in, hist out. |
| 3 | <code>i = cuda.grid(1)</code> | Start index of this thread. |
| 4 | <code>stride = cuda.gridsize(1)</code> | Total threads in the grid, the jump of the grid stride loop. |
| 5 | <code>while i &lt; data.size:</code> | Loop over this thread's share of the data. |
| 6 | <code>cuda.atomic.add(hist, data[i], 1)</code> | Add 1 to the bin whose number is the byte value. Atomic, so no update is lost. |
| 7 | <code>i += stride</code> | Jump to this thread's next element. |
| 9 | <code>hist = cuda.to_device(np.zeros(BINS, dtype=np.int32))</code> | A zeroed histogram on the GPU. |
| 10 | <code>blocks = 30 if SIM else 1024</code> | A fixed number of blocks. The grid stride loop covers any N with it. |
| 11 | <code>hist_global[blocks, TPB](d_data, hist)</code> | Launch. |
| 12 | <code>print(&quot;global atomics correct:&quot;, np.array_equal(hist.copy_to_host(), ref))</code> | array_equal needs every count to match exactly. Integer atomics give exact results. |

## 2. Privatisation: one histogram per block in shared memory

Each block counts into its own 256 bins in shared memory, where atomics are much cheaper and only the threads of one
block compete. At the end each block adds its 256 counts to the global histogram, so the global atomics drop from N to
256 per block.

In [ ]:
@cuda.jit
def hist_private(data, hist):
    local = cuda.shared.array(BINS, int32)
    t = cuda.threadIdx.x
    for b in range(t, BINS, cuda.blockDim.x):
        local[b] = 0
    cuda.syncthreads()
    i = cuda.grid(1)
    stride = cuda.gridsize(1)
    while i < data.size:
        cuda.atomic.add(local, data[i], 1)
        i += stride
    cuda.syncthreads()
    for b in range(t, BINS, cuda.blockDim.x):
        if local[b] > 0:
            cuda.atomic.add(hist, b, local[b])

hist = cuda.to_device(np.zeros(BINS, dtype=np.int32))
hist_private[blocks, TPB](d_data, hist)
print("private histograms correct:", np.array_equal(hist.copy_to_host(), ref))
print("global atomics: before", N, " after at most", blocks * BINS)

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>@cuda.jit</code> | Compile as a kernel. |
| 2 | <code>def hist_private(data, hist):</code> | Same inputs. |
| 3 | <code>local = cuda.shared.array(BINS, int32)</code> | The block's own 256 counters in shared memory. |
| 4 | <code>t = cuda.threadIdx.x</code> | Thread index in the block. |
| 5 | <code>for b in range(t, BINS, cuda.blockDim.x):</code> | Zero the counters. With 128 threads each thread clears 2 bins, with 256 threads one bin. |
| 6 | <code>local[b] = 0</code> | Set this bin to zero. |
| 7 | <code>cuda.syncthreads()</code> | All bins must be zero before anyone counts. |
| 8 | <code>i = cuda.grid(1)</code> | Start index for the grid stride loop. |
| 9 | <code>stride = cuda.gridsize(1)</code> | Jump size. |
| 10 | <code>while i &lt; data.size:</code> | Loop over this thread's elements. |
| 11 | <code>cuda.atomic.add(local, data[i], 1)</code> | Count into shared memory. Still atomic, because threads of the same block can see the same byte. |
| 12 | <code>i += stride</code> | Next element. |
| 13 | <code>cuda.syncthreads()</code> | All counting must finish before the block reads its counters. |
| 14 | <code>for b in range(t, BINS, cuda.blockDim.x):</code> | Each thread takes a few bins again. |
| 15 | <code>if local[b] &gt; 0:</code> | Skip empty bins, which saves atomics. |
| 16 | <code>cuda.atomic.add(hist, b, local[b])</code> | Add the block's count to the global histogram. One atomic per bin per block. |
| 18 | <code>hist = cuda.to_device(np.zeros(BINS, dtype=np.int32))</code> | A fresh zeroed histogram. |
| 19 | <code>hist_private[blocks, TPB](d_data, hist)</code> | Launch with the same shape. |
| 20 | <code>print(&quot;private histograms correct:&quot;, np.array_equal(hist.copy_to_host(), ref))</code> | Check against NumPy. |
| 21 | <code>print(&quot;global atomics: before&quot;, N, &quot; after at most&quot;, blocks * BINS)</code> | The saving in global atomics. On a GPU this is where the speedup comes from. |

## 3. The Blelloch scan in one block

An exclusive scan turns [3, 1, 7, 0] into [0, 3, 4, 11]: each output is the sum of everything before it.
The up sweep builds partial sums in a tree. Then we put 0 at the root, and the down sweep pushes sums back down.
Total work is 2(n − 1) additions, compared with n log2 n for the simpler Hillis and Steele scan.

In [ ]:
SCAN_N = 16 if SIM else 1024

@cuda.jit
def blelloch(x, out):
    s = cuda.shared.array(SCAN_N, int32)
    t = cuda.threadIdx.x
    s[2 * t] = x[2 * t]
    s[2 * t + 1] = x[2 * t + 1]
    offset = 1
    d = SCAN_N // 2
    while d > 0:
        cuda.syncthreads()
        if t < d:
            a = offset * (2 * t + 1) - 1
            b = offset * (2 * t + 2) - 1
            s[b] += s[a]
        offset *= 2
        d //= 2
    if t == 0:
        s[SCAN_N - 1] = 0
    d = 1
    while d < SCAN_N:
        offset //= 2
        cuda.syncthreads()
        if t < d:
            a = offset * (2 * t + 1) - 1
            b = offset * (2 * t + 2) - 1
            tmp = s[a]
            s[a] = s[b]
            s[b] += tmp
        d *= 2
    cuda.syncthreads()
    out[2 * t] = s[2 * t]
    out[2 * t + 1] = s[2 * t + 1]

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>SCAN_N = 16 if SIM else 1024</code> | Scan size. One block of SCAN_N ÷ 2 threads handles it, each thread owns two values. |
| 3 | <code>@cuda.jit</code> | Compile as a kernel. |
| 4 | <code>def blelloch(x, out):</code> | x in, out gets the exclusive scan. |
| 5 | <code>s = cuda.shared.array(SCAN_N, int32)</code> | The working array in shared memory. |
| 6 | <code>t = cuda.threadIdx.x</code> | Thread index. |
| 7 | <code>s[2 * t] = x[2 * t]</code> | Each thread loads two values: positions 2t ... |
| 8 | <code>s[2 * t + 1] = x[2 * t + 1]</code> | ... and 2t + 1. |
| 9 | <code>offset = 1</code> | Distance between the two children in the tree. It doubles each level of the up sweep. |
| 10 | <code>d = SCAN_N // 2</code> | Number of working threads on the first level: half the elements. |
| 11 | <code>while d &gt; 0:</code> | Up sweep: one level per pass, until one thread is left. |
| 12 | <code>cuda.syncthreads()</code> | Wait for the previous level (and for the loads on the first pass). |
| 13 | <code>if t &lt; d:</code> | Only the first d threads work on this level. |
| 14 | <code>a = offset * (2 * t + 1) - 1</code> | Index of the left child. |
| 15 | <code>b = offset * (2 * t + 2) - 1</code> | Index of the right child, which becomes the parent. |
| 16 | <code>s[b] += s[a]</code> | The parent gets the sum of both children. |
| 17 | <code>offset *= 2</code> | Children are twice as far apart on the next level. |
| 18 | <code>d //= 2</code> | Half as many threads work on the next level. |
| 19 | <code>if t == 0:</code> | After the up sweep the last slot holds the total. |
| 20 | <code>s[SCAN_N - 1] = 0</code> | Replace it by 0, the identity of addition. This is what makes the scan exclusive. |
| 21 | <code>d = 1</code> | Down sweep: start with one working thread. |
| 22 | <code>while d &lt; SCAN_N:</code> | Go until every pair has been visited. |
| 23 | <code>offset //= 2</code> | Children get closer on each level going down. |
| 24 | <code>cuda.syncthreads()</code> | Wait for the previous level. |
| 25 | <code>if t &lt; d:</code> | Only d threads work. |
| 26 | <code>a = offset * (2 * t + 1) - 1</code> | Left child index. |
| 27 | <code>b = offset * (2 * t + 2) - 1</code> | Right child index, which holds the value coming down from above. |
| 28 | <code>tmp = s[a]</code> | Keep the left child's sum. |
| 29 | <code>s[a] = s[b]</code> | The left child gets what came from above. |
| 30 | <code>s[b] += tmp</code> | The right child gets what came from above plus the old left sum. |
| 31 | <code>d *= 2</code> | Twice as many threads on the next level. |
| 32 | <code>cuda.syncthreads()</code> | Wait for the last level before writing out. |
| 33 | <code>out[2 * t] = s[2 * t]</code> | Each thread writes its two results. |
| 34 | <code>out[2 * t + 1] = s[2 * t + 1]</code> | Second result. |

In [ ]:
xs = np.random.default_rng(4).integers(0, 10, SCAN_N).astype(np.int32)
d_out = cuda.device_array(SCAN_N, dtype=np.int32)
blelloch[1, SCAN_N // 2](cuda.to_device(xs), d_out)
scan = d_out.copy_to_host()
print("input :", xs[:8], "...")
print("output:", scan[:8], "...")
print("matches NumPy:", np.array_equal(scan, np.concatenate(([0], np.cumsum(xs)[:-1]))))
print("additions: Blelloch", 2 * (SCAN_N - 1), " Hillis and Steele", SCAN_N * int(np.log2(SCAN_N)) - (SCAN_N - 1))

Line by line, what this cell does and why:

| Line | Code | What and why |
|---|---|---|
| 1 | <code>xs = np.random.default_rng(4).integers(0, 10, SCAN_N).astype(np.int32)</code> | Small random whole numbers, easy to check by eye. |
| 2 | <code>d_out = cuda.device_array(SCAN_N, dtype=np.int32)</code> | Space for the result. |
| 3 | <code>blelloch[1, SCAN_N // 2](cuda.to_device(xs), d_out)</code> | One block with SCAN_N ÷ 2 threads. |
| 4 | <code>scan = d_out.copy_to_host()</code> | Copy the result back. |
| 5 | <code>print(&quot;input :&quot;, xs[:8], &quot;...&quot;)</code> | The first inputs. |
| 6 | <code>print(&quot;output:&quot;, scan[:8], &quot;...&quot;)</code> | The first outputs. Each is the sum of all the inputs before it. |
| 7 | <code>print(&quot;matches NumPy:&quot;, np.array_equal(scan, np.concatenate(([0], np.cumsum(xs)[:-1]))))</code> | The exclusive scan in NumPy: a 0 followed by the running sum without its last value. |
| 8 | <code>print(&quot;additions: Blelloch&quot;, 2 * (SCAN_N - 1), &quot; Hillis and Steele&quot;, SCAN_N * int(np.log2(SCAN_N)) - (SCAN_N - 1))</code> | Work of the two scans. Hillis and Steele does n log2 n − (n − 1) additions, Blelloch 2(n − 1). |

## 4. Try these

1. Turn the exclusive scan into an inclusive one with one extra line.
2. Use the scan to do stream compaction: keep only the even numbers of an array, in order.
3. Time hist_global and hist_private on a GPU for data that is all the same byte. Explain why global atomics get much slower.